# Model Building

### Imports

In [1]:
# architecture: char-cnn + word-embed -> concat -> bilstm -> crf
# everything uses tf.keras model subclassing so layers are explicit and testable.
# order: charCNN layer -> word embed layer -> bilstm layer -> crf layer -> full model -> smoke test
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Sequential
from tensorflow.keras.layers import Layer, Embedding, Conv1D, GlobalMaxPooling1D, Dropout, Bidirectional, LSTM, Dense, BatchNormalization, ReLU
from pathlib import Path

print(f"tensorflow {tf.__version__}")

tensorflow 2.21.0


In [2]:
import sys
sys.path.insert(0, '../src')

from preprocessing.preprocess import DatasetBuilder

### Build Dataset

In [3]:
# build from the fixed dataset — vocabs are built strictly on train split
builder = DatasetBuilder(
    max_len=100,
    max_word_len=15,
    min_word_freq=2,   # words appearing < 2 times in train → [UNK]
    train_pct=0.8,
    val_pct=0.1,
    seed=42,
    verbose=True,
)

train_ds, val_ds, test_ds = builder.build_from_file(
    '../data/dataset_fixed.json',
    batch_size=32,
)

aligning BIO tags: 100%|██████████| 13226/13226 [00:00<00:00, 23547.31it/s]


split totals -> train: 10580, val: 1322, test: 1324
vocab sizes -> word: 14669, char: 118, tag: 9


### Save Vocabs

In [4]:
Path('../data/processed').mkdir(parents=True, exist_ok=True)

builder.save_vocabs('../data/processed/')

for f in Path('../data/processed').iterdir():
    print(f.name, '—', f.stat().st_size, 'bytes')

saved vocabs to ../data/processed/
char2idx.json — 1502 bytes
tag2idx.json — 157 bytes
word2idx.json — 283622 bytes


### Load Vocabs

In [5]:
with open('../data/processed/word2idx.json', encoding='utf-8') as f:
    word2idx = json.load(f)
with open('../data/processed/char2idx.json', encoding='utf-8') as f:
    char2idx = json.load(f)
with open('../data/processed/tag2idx.json', encoding='utf-8') as f:
    tag2idx = json.load(f)
    
idx2tag = {v: k for k, v in tag2idx.items()}

In [6]:
# hyperparams
VOCAB_SIZE     = len(word2idx)       # word vocab
CHAR_VOCAB     = len(char2idx)       # character vocab
NUM_TAGS       = len(tag2idx)        # output classes (BIO tags + PAD)
MAX_LEN        = 100                 # max tokens per sequence
MAX_WORD_LEN   = 15                  # max chars per token

WORD_EMBED_DIM = 128                 # word embedding size
CHAR_EMBED_DIM = 32                  # char embedding size (input to CNN)
CHAR_CNN_DIM   = 128                 # char CNN output size (after global max pool)
BILSTM_UNITS   = 256                 # units per direction -> output is 512
DROPOUT        = 0.3

print(f"vocab_size={VOCAB_SIZE}  char_vocab={CHAR_VOCAB}  num_tags={NUM_TAGS}")
print(f"combined input to bilstm: {WORD_EMBED_DIM + CHAR_CNN_DIM}")

vocab_size=14669  char_vocab=118  num_tags=9
combined input to bilstm: 256


### Char-CNN Embedding Layer

In [7]:
# the char cnn is used to answer "what does this token look like at the character level?"
# this is crucial for prices like Rp133k, 335k-an, cenggo, and etc...
# those price patterns above are OOV at word level but has learnable character patterns
#
# design:
#   input:  (batch, max_len, max_word_len)          — char indices per token
#   output: (batch, max_len, CHAR_CNN_DIM)           — one vector per token
#
# we reshape to (batch*max_len, max_word_len) to run the CNN over all tokens
# in parallel, then reshape back. this is the standard trick.
#
# two conv blocks with different kernel sizes to capture trigram and 5-gram patterns.
# global max pool collapses the char dimension → fixed-size vector regardless of word length.
class CharCNNEmbedding(Layer):
    def  __init__(self, char_vocab, char_embed_dim, cnn_filters, kernel_sizes, dropout, **kwargs):
        super().__init__(**kwargs)
        # char_embed_dim: dimension of each character's embedding
        # cnn_filters: list of filter counts, one per conv block e.g. [64, 128]
        # kernel_sizes: list of kernel sizes matching cnn_filters e.g. [3, 5]
        
        self.char_embedding = Embedding(input_dim=char_vocab,
                                        output_dim=char_embed_dim,
                                        # mask_zero=True, # zero-index is PAD, masking propagates
                                        name='char_embed')
        
        # prallel conv blocks
        self.conv_blocks = []
        for filters, ksize in zip(cnn_filters, kernel_sizes):
            block = Sequential([Conv1D(filters, ksize, padding='same', use_bias=False),
                                BatchNormalization(),
                                ReLU()], name=f'conv_{ksize}')
            self.conv_blocks.append(block)
        
        # global max pool per conv block then concat -> CHAR_CNN_DIM total
        self.pool = GlobalMaxPooling1D()
        self.dropout = Dropout(dropout)
        
        # project concatenated conv outputs to CHAR_CNN_DIM
        total_filters = sum(cnn_filters)
        self.proj = Dense(CHAR_CNN_DIM, use_bias=False, name='char_proj')
    
    def call(self, char_inputs, training=False):
        # char inputs (batch, seq_len, word_len)
        batch = tf.shape(char_inputs)[0]
        seq = tf.shape(char_inputs)[1]
        
        # flatten sequences so CNN sees (batch*seq_len, word_len)
        flat = tf.reshape(char_inputs, (-1, MAX_WORD_LEN)) # (batch * seq_len, word_len)
        x = self.char_embedding(flat) # (batch * seq_len, word_len, char_embed_dim)
        
        # conv and pool
        pooled = []
        for conv in self.conv_blocks:
            h = conv(x, training=training) # (batch * seq_len, word_len, filters)
            h = self.pool(h) # (batch * seq_len, filters)
            pooled.append(h)
        
        # concatenate all conv outputs along filter axis
        x = tf.concat(pooled, axis=-1) # (batch_size * seq_len, sum_filters)
        x = self.dropout(x, training=training)
        x = self.proj(x) # (batch_size * seq_len, CHAR_CNN_DIM)
        
        # restore sequence shape
        x = tf.reshape(x, (batch, seq, CHAR_CNN_DIM)) # (batch_size, seq_len, CHAR_CNN_DIM)
        return x
    
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], self.proj.units)

#### Test Char CNN

In [8]:
# use a dummy batch that mimics what DatasetBuilder produces.
dummy_chars = tf.zeros((4, MAX_LEN, MAX_WORD_LEN), dtype=tf.int32)  # batch=4

char_cnn = CharCNNEmbedding(
    char_vocab=CHAR_VOCAB,
    char_embed_dim=CHAR_EMBED_DIM,
    cnn_filters=[64, 128],
    kernel_sizes=[3, 5],
    dropout=DROPOUT,
)

out = char_cnn(dummy_chars, training=False)

assert out.shape == (4, MAX_LEN, CHAR_CNN_DIM), f"unexpected shape: {out.shape}"
print(f"char cnn output shape: {out.shape}")  # expected (4, 100, 128)
print(f"trainable params: {sum(np.prod(v.shape) for v in char_cnn.trainable_variables):,}")

char cnn output shape: (4, 100, 128)
trainable params: 55,360


### Word Embedding Layer

In [9]:
# word embeddings modes controlled by trainable:
#   trainable=True  -> learned from scratch
#   trainable=False -> frozen pre-trained weights
#
# we support loading pre-trained weights via load_pretrained() below.
# mask_zero=True propagates padding masks to the bilstm
class WordEmbedding(Layer):
    def __init__(self, vocab_size, embed_dim, dropout, trainable_embed=True, **kwargs):
        super().__init__(**kwargs)
        self.embedding = Embedding(input_dim=vocab_size,
                                   output_dim=embed_dim,
                                   mask_zero=False,
                                   trainable=trainable_embed,
                                   name='word_embed')
        self.dropout = Dropout(dropout)
    
    def call(self, word_inputs, training=False):
        # word_inputs: (batch, seq_len)  → int indices
        x = self.embedding(word_inputs)      # (batch, seq_len, embed_dim)
        return self.dropout(x, training=training)

    def load_pretrained(self, embedding_matrix):
        # embedding_matrix = np array of shape (vocab_size, embed_dim)
        self.embedding.set_weights([embedding_matrix])
        print(f"loaded pretrained weights: {embedding_matrix.shape}")
        
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], self.embedding.output_dim)

#### Test Word Embedding Layer

In [10]:
# unit test
dummy_words = tf.zeros((4, MAX_LEN), dtype=tf.int32)
word_emb = WordEmbedding(VOCAB_SIZE, WORD_EMBED_DIM, DROPOUT)
out = word_emb(dummy_words, training=False)

assert out.shape == (4, MAX_LEN, WORD_EMBED_DIM), f"unexpected shape: {out.shape}"
print(f"word embed output shape: {out.shape}")  # expected (4, 100, 128)

word embed output shape: (4, 100, 128)


### Concat + BiLSTM Encoder

In [ ]:
# the bilstm sees the concatenated [word_embed; char_cnn_vec] at each timestep.
# return_sequences=True is mandatory — we need a tag at every position, not just the last.
# we stack two bilstm layers, the first returns sequences into the second.
#
# note: tf.keras Bidirectional wraps the LSTM and concatenates fwd+bwd outputs,
# so a LSTM(units=256) wrapped in Bidirectional -> output dim is 512.
class BiLSTMEncoder(Layer):
    def __init__(self, units, dropout, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True
        
        # first lstm layer returns sequences into the second
        self.bilstm1 = Bidirectional(LSTM(units, return_sequences=True, recurrent_dropout=dropout),
                                     name='bilstm_1')
        
        # second lstm layer
        self.bilstm2 = Bidirectional(LSTM(units, return_sequences=True, recurrent_dropout=dropout),
                                     name='bilstm_2')
        
        self.dropout = Dropout(dropout)
        
    def call(self, x, mask=None, training=False):
        # x: (batch, seq_len, word_embed_dim + char_cnn_dim)
        x = self.bilstm1(x, mask=mask, training=training)  # (b, s, units*2)
        x = self.dropout(x, training=training)
        x = self.bilstm2(x, mask=mask, training=training)  # (b, s, units*2)
        return x
    
    def compute_output_shape(self, input_shape):
        return self.bilstm2.compute_output_shape(input_shape)

#### Test BiLSTM Encoder

In [12]:
dummy_concat = tf.zeros((4, MAX_LEN, WORD_EMBED_DIM + CHAR_CNN_DIM))
bilstm = BiLSTMEncoder(BILSTM_UNITS, DROPOUT)
out = bilstm(dummy_concat, training=False)

assert out.shape == (4, MAX_LEN, BILSTM_UNITS * 2), f"unexpected: {out.shape}"
print(f"bilstm output shape: {out.shape}") 

bilstm output shape: (4, 100, 512)


### CRF Layer

In [13]:
# a linear-chain CRF adds a transition matrix on top of the bilstm emissions.
# the layer holds one learnable parameter: transition_params (num_tags, num_tags).
# transition_params[i, j] = score of transitioning from tag i to tag j.
#
# at training time we need:
#   1. score of the true tag path      (sum of emissions + transitions along ground truth)
#   2. log partition function Z        (sum over ALL possible paths, computed via forward alg)
#   nll = log(Z) - score               (we minimize this)
#
# at inference time we run viterbi decoding to find the highest-scoring tag sequence.
class CRFLayer(Layer):
    def __init__(self, num_tags, **kwargs):
        super().__init__(**kwargs)
        self.num_tags = num_tags
        
        # transition_params[i, j]: score of going from tag i → tag j
        # initialized to zeros; the model learns which transitions are (im)possible
        self.transition_params = self.add_weight(name='transition_params',
                                                 shape=(self.num_tags, self.num_tags),
                                                 initializer='zeros',
                                                 trainable=True)
        self.built = True
        
    def build(self, input_shape):
        super().build(input_shape)
        
    def call(self, emissions, mask=None):
        # at inference we run viterbi and return the best tag sequence
        # at training the loss function calls log_likelihood directly
        # emissions shape: (batch, seq_len, num_tags)
        return self.viterbi_decode(emissions, mask)
    
    def log_likelihood(self, emissions, tag_indices, mask):
        """
        compute the CRF log-likelihood for a batch.
        emissions:   (batch, seq_len, num_tags)
        tag_indices: (batch, seq_len)
        mask:        (batch, seq_len)

        returns scalar: mean negative log-likelihood over the batch.
        """
        batch_size = tf.shape(emissions)[0]
        seq_len = tf.shape(emissions)[1]
        
        # score of the true path
        true_score = self._score_sequence(emissions, tag_indices, mask)
        
        # log partition function
        log_Z = self._forward_algorithm(emissions, mask)
        
        # nll per sample
        nll = log_Z - true_score
        return tf.reduce_mean(nll) # mean over batch
    
    def _score_sequence(self, emissions, tag_indices, mask):
        """sum emission scores + transition scores along the true path."""
        batch_size = tf.shape(emissions)[0]
        seq_len    = tf.shape(emissions)[1]
        
        # emission scores: (batch, seq_len)
        batch_idx = tf.tile(tf.expand_dims(tf.range(batch_size), 1), [1, seq_len])
        seq_idx = tf.tile(tf.expand_dims(tf.range(seq_len), 0), [batch_size, 1])
        indices = tf.stack([batch_idx, seq_idx, tag_indices], axis=2)
        
        emit_scores = tf.gather_nd(emissions, indices) # (b, s)
        
        # transition scores: transition_params[y_{t-1}, y_t] for t in 1..T
        # slice off first and last to align consecutive pairs
        trans_scores = tf.gather_nd(self.transition_params,
                                    tf.stack([tag_indices[:, :-1], tag_indices[:, 1:]], axis=2)) # (b, s-1)
        
        # apply mask
        mask_f = tf.cast(mask, tf.float32)
        total = tf.reduce_sum(emit_scores * mask_f, axis=1)
        total += tf.reduce_sum(trans_scores * mask_f[:, 1:], axis=1)
        return total # (batch,)
    
    def _forward_algorithm(self, emissions, mask):
        """
        log-space forward algorithm to compute log Z (partition function).
        uses log-sum-exp for numerical stability.
        """
        seq_len = tf.shape(emissions)[1]
        mask_f  = tf.cast(mask, tf.float32)

        # initialize: alpha[0] = emissions at t=0
        alphas = emissions[:, 0, :] # (batch, num_tags)

        for t in tf.range(1, seq_len):
            # expand for broadcasting: (batch, num_tags, 1) + (num_tags, num_tags)
            # transition_scores[i, j] = alpha[i] + transition[i, j] + emission[j]
            emit_t  = emissions[:, t, :] # (batch, num_tags)
            trans_t = tf.expand_dims(alphas, 2) + self.transition_params  # (batch, num_tags, num_tags)
            # log-sum-exp over previous tags
            new_alphas = tf.reduce_logsumexp(trans_t, axis=1) + emit_t # (batch, num_tags)

            # only update alphas for non-padding positions
            mask_t = tf.expand_dims(mask_f[:, t], 1) # (batch, 1)
            alphas = new_alphas * mask_t + alphas * (1 - mask_t)

        # final log Z: log-sum-exp over last alphas
        return tf.reduce_logsumexp(alphas, axis=1) # (batch,)
    
    def viterbi_decode(self, emissions, mask):
        """
        viterbi decoding: find highest-scoring tag sequence.
        returns (tag_ids, viterbi_score) both shape (batch,) and (batch, seq_len).
        """
        seq_len = tf.shape(emissions)[1]
        mask_f  = tf.cast(mask, tf.float32)

        viterbi  = emissions[:, 0, :] # (batch, num_tags)
        backpointers = []

        for t in tf.range(1, seq_len):
            emit_t = emissions[:, t, :] # (batch, num_tags)
            # scores for all prev_tag → cur_tag transitions
            trans_scores = tf.expand_dims(viterbi, 2) + self.transition_params  # (b, T, T)
            best_prev    = tf.argmax(trans_scores, axis=1) # (b, num_tags)
            best_scores  = tf.reduce_max(trans_scores, axis=1) + emit_t

            mask_t = tf.expand_dims(mask_f[:, t], 1)
            viterbi = best_scores * mask_t + viterbi * (1 - mask_t)
            backpointers.append(best_prev)

        # backtrace
        best_last = tf.argmax(viterbi, axis=1) # (batch,)
        best_path = [best_last]

        for bp in reversed(backpointers):
            # gather best previous tag at each position
            batch_size = tf.shape(emissions)[0]
            idx = tf.stack([tf.range(tf.cast(batch_size, tf.int64), dtype=tf.int64), best_path[-1]], axis=1)
            best_last = tf.gather_nd(tf.cast(bp, tf.int64), idx)
            best_path.append(best_last)

        best_path = list(reversed(best_path))
        best_path = tf.stack(best_path, axis=1) # (batch, seq_len)
        return best_path
    
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1])

#### Test CRF Layer

In [14]:
dummy_emissions = tf.random.normal((4, MAX_LEN, NUM_TAGS))
dummy_mask      = tf.ones((4, MAX_LEN), dtype=tf.bool)
dummy_tags      = tf.zeros((4, MAX_LEN), dtype=tf.int32)

crf = CRFLayer(NUM_TAGS)
crf.build((None, MAX_LEN, NUM_TAGS))

nll   = crf.log_likelihood(dummy_emissions, dummy_tags, dummy_mask)
preds = crf.viterbi_decode(dummy_emissions, dummy_mask)

print(f"crf nll (dummy): {nll:.4f}")
print(f"viterbi output shape: {preds.shape}")
print(f"transition matrix shape: {crf.transition_params.shape}")

crf nll (dummy): 265.1739
viterbi output shape: (4, 100)
transition matrix shape: (9, 9)


### NERModel

In [15]:
class BillNERModel(Model):
    def __init__(self, vocab_size, char_vocab, num_tags,
                 word_embed_dim=128, char_embed_dim=32, char_cnn_dim=128,
                 bilstm_units=256, dropout=0.3, **kwargs):
        super().__init__(**kwargs)

        self.word_embedding = WordEmbedding(vocab_size, word_embed_dim, dropout)

        self.char_cnn = CharCNNEmbedding(
            char_vocab=char_vocab,
            char_embed_dim=char_embed_dim,
            cnn_filters=[64, 128],
            kernel_sizes=[3, 5],
            dropout=dropout,
        )

        self.bilstm = BiLSTMEncoder(bilstm_units, dropout)

        # project bilstm output to tag space before CRF
        self.emission_proj = layers.Dense(num_tags, name='emission_proj')

        self.crf = CRFLayer(num_tags)

    def call(self, inputs, training=False):
        word_in = inputs['word_inputs']   # (batch, seq_len)
        char_in = inputs['char_inputs']   # (batch, seq_len, word_len)

        # build mask from word inputs: True where word_idx > 0 (non-padding)
        mask = tf.cast(word_in > 0, tf.bool)   # (batch, seq_len)

        # embed
        word_vec = self.word_embedding(word_in, training=training) # (b, s, word_embed)
        char_vec = self.char_cnn(char_in, training=training) # (b, s, char_cnn_dim)

        # concat and encode
        x = tf.concat([word_vec, char_vec], axis=-1) # (b, s, word+char)
        x = self.bilstm(x, mask=mask, training=training) # (b, s, bilstm*2)

        # project to tag emissions
        emissions = self.emission_proj(x) # (b, s, num_tags)
        return emissions, mask

    def decode(self, inputs):
        """inference: returns viterbi tag sequence"""
        emissions, mask = self(inputs, training=False)
        return self.crf.viterbi_decode(emissions, mask)

### Test NER Model

In [16]:
optimizer = keras.optimizers.Adam(learning_rate=1e-3, clipnorm=5.0)

model = BillNERModel(
    vocab_size=VOCAB_SIZE,
    char_vocab=CHAR_VOCAB,
    num_tags=NUM_TAGS,
    word_embed_dim=WORD_EMBED_DIM,
    char_embed_dim=CHAR_EMBED_DIM,
    char_cnn_dim=CHAR_CNN_DIM,
    bilstm_units=BILSTM_UNITS,
    dropout=DROPOUT,
)

@tf.function
def train_step(batch_inputs, batch_tags):
    with tf.GradientTape() as tape:
        emissions, mask = model(batch_inputs, training=True)
        loss = model.crf.log_likelihood(
            emissions,
            tf.cast(batch_tags, tf.int32),
            mask,
        )
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

@tf.function
def val_step(batch_inputs, batch_tags):
    emissions, mask = model(batch_inputs, training=False)
    loss = model.crf.log_likelihood(
        emissions,
        tf.cast(batch_tags, tf.int32),
        mask,
    )
    return loss

#### Model Summary

In [17]:
# create the model
model = BillNERModel(
    vocab_size=VOCAB_SIZE,
    char_vocab=CHAR_VOCAB,
    num_tags=NUM_TAGS,
    word_embed_dim=WORD_EMBED_DIM,
    char_embed_dim=CHAR_EMBED_DIM,
    char_cnn_dim=CHAR_CNN_DIM,
    bilstm_units=BILSTM_UNITS,
    dropout=DROPOUT,
)

# run a dummy batch to natively build all nested layers
dummy_batch = {
    'word_inputs': tf.zeros((1, MAX_LEN), dtype=tf.int32),
    'char_inputs': tf.zeros((1, MAX_LEN, MAX_WORD_LEN), dtype=tf.int32)
}
model(dummy_batch)
model.summary()

Model: "bill_ner_model_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ word_embedding_2                │ (1, 100, 128)          │     1,877,632 │
│ (WordEmbedding)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ char_cnn_embedding_2            │ (1, 100, 128)          │        55,744 │
│ (CharCNNEmbedding)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_lstm_encoder_2               │ (1, 100, 512)          │     2,625,536 │
│ (BiLSTMEncoder)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ emission_proj (Dense)           │ (1, 100, 9)            │         4,617 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ crf_layer_2 (CRFLayer)          │ ?                      │            81 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,563,610 (17.41 MB)

 Trainable params: 4,563,226 (17.41 MB)

 Non-trainable params: 384 (1.50 KB)

#### Test On 1 Training Step

In [18]:
# grab one batch
for batch_inputs, batch_tags in train_ds.take(1):
    print("word_inputs shape :", batch_inputs['word_inputs'].shape)
    print("char_inputs shape :", batch_inputs['char_inputs'].shape)
    print("tags shape        :", batch_tags.shape)

    # forward pass
    emissions, mask = model(batch_inputs, training=False)
    print("emissions shape   :", emissions.shape)   # (batch, max_len, num_tags)
    print("mask shape        :", mask.shape)

    # one training step
    loss = train_step(batch_inputs, batch_tags)
    print(f"loss after 1 step : {loss:.4f}")

    # check no None gradients
    with tf.GradientTape() as tape:
        e, m = model(batch_inputs, training=True)
        l    = model.crf.log_likelihood(e, tf.cast(batch_tags, tf.int32), m)
    grads = tape.gradient(l, model.trainable_variables)
    none_grads = [v.name for v, g in zip(model.trainable_variables, grads) if g is None]
    print(f"none gradients    : {none_grads if none_grads else 'none — all good'}")

    # viterbi decode
    preds = model.decode(batch_inputs)
    print(f"viterbi output    : {preds.shape}")

word_inputs shape : (32, 100)
char_inputs shape : (32, 100, 15)
tags shape        : (32, 100)
emissions shape   : (32, 100, 9)
mask shape        : (32, 100)
loss after 1 step : 110.6685
none gradients    : none — all good
viterbi output    : (32, 100)


### Test From Modularized Code

In [20]:
from models.ner_model import BillNERModel

# create the model
model = BillNERModel(
    vocab_size=VOCAB_SIZE,
    char_vocab=CHAR_VOCAB,
    num_tags=NUM_TAGS,
    word_embed_dim=WORD_EMBED_DIM,
    char_embed_dim=CHAR_EMBED_DIM,
    char_cnn_dim=CHAR_CNN_DIM,
    bilstm_units=BILSTM_UNITS,
    dropout=DROPOUT,
)

# run a dummy batch to natively build all nested layers
dummy_batch = {
    'word_inputs': tf.zeros((1, MAX_LEN), dtype=tf.int32),
    'char_inputs': tf.zeros((1, MAX_LEN, MAX_WORD_LEN), dtype=tf.int32)
}
model(dummy_batch)
model.summary()

# grab one batch
for batch_inputs, batch_tags in train_ds.take(1):
    print("word_inputs shape :", batch_inputs['word_inputs'].shape)
    print("char_inputs shape :", batch_inputs['char_inputs'].shape)
    print("tags shape        :", batch_tags.shape)

    # forward pass
    emissions, mask = model(batch_inputs, training=False)
    print("emissions shape   :", emissions.shape)   # (batch, max_len, num_tags)
    print("mask shape        :", mask.shape)

    # one training step
    loss = train_step(batch_inputs, batch_tags)
    print(f"loss after 1 step : {loss:.4f}")

    # check no None gradients
    with tf.GradientTape() as tape:
        e, m = model(batch_inputs, training=True)
        l    = model.crf.log_likelihood(e, tf.cast(batch_tags, tf.int32), m)
    grads = tape.gradient(l, model.trainable_variables)
    none_grads = [v.name for v, g in zip(model.trainable_variables, grads) if g is None]
    print(f"none gradients    : {none_grads if none_grads else 'none — all good'}")

    # viterbi decode
    preds = model.decode(batch_inputs)
    print(f"viterbi output    : {preds.shape}")

Model: "bill_ner_model_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ word_embedding_3                │ (1, 100, 128)          │     1,877,632 │
│ (WordEmbedding)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ char_cnn_embedding_3            │ (1, 100, 128)          │        55,744 │
│ (CharCNNEmbedding)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_lstm_encoder_3               │ (1, 100, 512)          │     2,625,536 │
│ (BiLSTMEncoder)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ emission_proj (Dense)           │ (1, 100, 9)            │         4,617 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ crf_layer_3 (CRFLayer)          │ ?                      │            81 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,563,610 (17.41 MB)

 Trainable params: 4,563,226 (17.41 MB)

 Non-trainable params: 384 (1.50 KB)

word_inputs shape : (32, 100)
char_inputs shape : (32, 100, 15)
tags shape        : (32, 100)
emissions shape   : (32, 100, 9)
mask shape        : (32, 100)
loss after 1 step : 69.7342
none gradients    : none — all good
viterbi output    : (32, 100)
